In [ ]:
// #r "./binaries/BoSSSpad.dll"
// #r "./binaries/LsTest.dll"
#r "BoSSSpad.dll"
#r "LsTest.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Solution.LevelSetTools;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Application.LsTest;
Init();

# Load sessions

Init Workflowmanagement and Project

In [ ]:
BoSSSshell.WorkflowMgm.Init("SwirlingFlow_SpatialConvergence");
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination);
sessions

In [ ]:
using System.IO;

In [ ]:
var tab = sessions.GetSessionTable();

In [ ]:
var Evos = tab.GetColumn<string>("id:Evo").Distinct();
Evos.ForEach(e => Console.WriteLine(e));

var tRes = tab.GetColumn<double>("id:dt").Distinct();
tRes.ForEach(e => Console.WriteLine(e));

var pRes = tab.GetColumn<int>("id:Degree").Distinct();
pRes.ForEach(e => Console.WriteLine(e));

var hRes = tab.GetColumn<int>("id:Res").Distinct();
hRes.ForEach(e => Console.WriteLine(e));

In [ ]:
List<Plot2Ddata> plts = new List<Plot2Ddata>();
foreach(double dt in tRes){
    foreach(string Evo in Evos){
        var plt = new Plot2Ddata().WithLogX().WithLogY();
        foreach(int p in pRes){
            // select and sort
            var sess = sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Evo"]) == Evo &&  Convert.ToDouble(s.KeysAndQueries["id:dt"]) == dt && Convert.ToInt32(s.KeysAndQueries["id:Degree"]) == p).ToArray();
            if(sess.Count() == 0) continue;
            Array.Sort(sess, (a,b) => Convert.ToDouble(a.KeysAndQueries["id:Res"]).CompareTo(Convert.ToDouble(b.KeysAndQueries["id:Res"])));
            sess.ForEach(s => Console.WriteLine(s.Name));

            var Ref = new BoSSS.Application.LsTest.LevelSetUnitTests.LevelSetSwirlingFlowTest(2, p, false, 0.0); // to get the "Phi" Functions etc.
            var RefC = BoSSS.Application.LsTest.LevelSetUnitTests.LSTstObj2CtrlObj(Ref, int.MaxValue, LevelSetEvolution.None, BoSSS.Solution.XdgTimestepping.LevelSetHandling.None, hRes.Max(), 0);
            var grd = RefC.GridFunc(); // evaluate on the same grid

            Func<ITimestepInfo, double> errorFunctional = ts => {
                var PhiDG = ts.Fields.Single(f => f.Identification == "PhiDG");
                var PhiCG = ts.Fields.Single(f => f.Identification == "Phi");

                var PhiDGFine = new LevelSet(new Basis(grd, PhiDG.Basis.Degree), "PhiDGFine");
                var PhiCGFine = new LevelSet(new Basis(grd, PhiCG.Basis.Degree), "PhiFine");
                PhiDGFine.ProjectFromForeignGrid(1.0, (SinglePhaseField)PhiDG);
                // PhiCGFine.ProjectFromForeignGrid(1.0, (SinglePhaseField)PhiCG);
                PhiCGFine.ProjectField(1.0, Ref.GetPhi()[0].Vectorize().SetTime(0.0));

                LevelSetTracker LsTrk = new LevelSetTracker((GridData)PhiCGFine.GridDat, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] { "A", "B" }, (ILevelSet)PhiCGFine);
                LsTrk.UpdateTracker(0.0);

                double err = 0.0;
                err = PhiDGFine.L2Error(Ref.GetPhi()[0].Vectorize().SetTime(0.0), 16, new BoSSS.Foundation.Quadrature.CellQuadratureScheme(true, LsTrk.Regions.GetNearFieldMask(1)));
                return err;
            };

            var errs = sess.Select(s => errorFunctional(s.Timesteps.Last())).ToArray();
            errs.ForEach(err => Console.WriteLine(err));

            var _plt = sess.ToGridConvergenceData(errorFunctional);
            _plt.dataGroups.Single().Name = " Degree: " + p;
            plt = plt.Merge(_plt);
        }        
        plts.Add(plt);
        plt.Title = "dt : " + dt + " Evo: " + Evo;
        plt.TitleFont = 30;
        plt.tmargin = 8;
        plt.ModFormat();
    }
}

In [ ]:
foreach(var plt in plts){
    display(plt.Title + " : ");
    plt.Regression().ForEach( g => display( g.Key + ", Slope : " + g.Value));
    display(plt.PlotNow());
    display(":::::::::::::::::::::::::::::::::::::::::");
}

In [ ]:
List<Plot2Ddata> plts2 = new List<Plot2Ddata>();
foreach(double dt in tRes){
    foreach(string Evo in Evos){
        try{
            var plt = new Plot2Ddata().WithLogX().WithLogY();
            foreach(int p in pRes){
                // select and sort
                var sess = sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Evo"]) == Evo &&  Convert.ToDouble(s.KeysAndQueries["id:dt"]) == dt && Convert.ToInt32(s.KeysAndQueries["id:Degree"]) == p).ToArray();
                if(sess.Count() == 0) continue;
                Array.Sort(sess, (a,b) => Convert.ToDouble(a.KeysAndQueries["id:Res"]).CompareTo(Convert.ToDouble(b.KeysAndQueries["id:Res"])));
                sess.ForEach(s => Console.WriteLine(s.Name));


                var _plt = sess.ToEstimatedGridConvergenceData("PhiDG");
                _plt.dataGroups.Single().Name = " Degree: " + p;
                plt = plt.Merge(_plt);
            }
            plts2.Add(plt);
            plt.Title = "dt : " + dt + " Evo: " + Evo;
            plt.TitleFont = 30;
            plt.tmargin = 8;
            plt.ModFormat();
        } catch {}
    }
}

In [ ]:
foreach(var plt in plts2){
    display(plt.Title + " : ");
    plt.Regression().ForEach( g => display( g.Key + ", Slope : " + g.Value));
    display(plt.PlotNow());
    display(":::::::::::::::::::::::::::::::::::::::::");
}

## Some potentially more meaningful metrics

1. Enclosed area, exact value : $A = \pi R^2$
2. Interface error, measured by $\epsilon = \sqrt{\int_\mathfrak{J} (\sqrt{(x-x_0)^2 + (y-y_0)^2} - R)^2 dS }$

$ R = 0.15, x_0 = 0.5, y_0 = 0.75$

In [ ]:
double R = 0.15;
double x0 = 0.5;
double y0 = 0.75;
double A_ex = Math.PI * R * R;

In [ ]:
using BoSSS.Foundation.Quadrature;

In [ ]:
List<Plot2Ddata> plts = new List<Plot2Ddata>();
foreach(double dt in tRes){
    foreach(string Evo in Evos){
        var plt = new Plot2Ddata().WithLogX().WithLogY();
        foreach(int p in pRes){
            // select and sort
            var sess = sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Evo"]) == Evo &&  Convert.ToDouble(s.KeysAndQueries["id:dt"]) == dt && Convert.ToInt32(s.KeysAndQueries["id:Degree"]) == p).ToArray();
            if(sess.Count() == 0) continue;
            Array.Sort(sess, (a,b) => Convert.ToDouble(a.KeysAndQueries["id:Res"]).CompareTo(Convert.ToDouble(b.KeysAndQueries["id:Res"])));
            sess.ForEach(s => Console.WriteLine(s.Name));

            var Ref = new BoSSS.Application.LsTest.LevelSetUnitTests.LevelSetSwirlingFlowTest(2, p, false, 0.0); // to get the "Phi" Functions etc.
            var RefC = BoSSS.Application.LsTest.LevelSetUnitTests.LSTstObj2CtrlObj(Ref, int.MaxValue, LevelSetEvolution.None, BoSSS.Solution.XdgTimestepping.LevelSetHandling.None, hRes.Max(), 0);
            var grd = RefC.GridFunc(); // evaluate on the same grid

            Func<ITimestepInfo, double> errorFunctional = ts => {
                var PhiCG = ts.Fields.Single(f => f.Identification == "Phi");

                var PhiCGFine = new LevelSet(new Basis(grd, PhiCG.Basis.Degree), "PhiFine");
                PhiCGFine.ProjectFromForeignGrid(1.0, (SinglePhaseField)PhiCG); // transfer all to the fine grid

                LevelSetTracker LsTrk = new LevelSetTracker((GridData)PhiCGFine.GridDat, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] { "A", "B" }, (ILevelSet)PhiCGFine);
                LsTrk.UpdateTracker(0.0);

                double A = 0.0;
                SpeciesId spcId = LsTrk.SpeciesIdS[0];
                var SchemeHelper = LsTrk.GetXDGSpaceMetrics(spcId, p).XQuadSchemeHelper;
                var vqs = SchemeHelper.GetVolumeQuadScheme(spcId);
                CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
                    vqs.Compile(grd.iGridData, p),
                    delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
                        EvalResult.SetAll(1.0);
                    },
                    delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
                        for (int i = 0; i < Length; i++)
                            A += ResultsOfIntegration[i, 0];
                    }
                ).Execute();
                double err = Math.Abs(A - A_ex);
                return err;
            };

            var errs = sess.Select(s => errorFunctional(s.Timesteps.Last())).ToArray();
            errs.ForEach(err => Console.WriteLine(err));

            var _plt = sess.ToGridConvergenceData(errorFunctional);
            _plt.dataGroups.Single().Name = " Degree: " + p;
            plt = plt.Merge(_plt);
        }        
        plts.Add(plt);
        plt.Title = "dt : " + dt + " Evo: " + Evo;
        plt.TitleFont = 30;
        plt.tmargin = 8;
        plt.ModFormat();
    }
}

In [ ]:
foreach(var plt in plts){
    display(plt.Title + " : ");
    plt.Regression().ForEach( g => display( g.Key + ", Slope : " + g.Value));
    display(plt.PlotNow());
    display(":::::::::::::::::::::::::::::::::::::::::");
}

In [ ]:
List<Plot2Ddata> plts = new List<Plot2Ddata>();
foreach(double dt in tRes){
    foreach(string Evo in Evos){
        var plt = new Plot2Ddata().WithLogX().WithLogY();
        foreach(int p in pRes){
            // select and sort
            var sess = sessions.Where(s => Convert.ToString(s.KeysAndQueries["id:Evo"]) == Evo &&  Convert.ToDouble(s.KeysAndQueries["id:dt"]) == dt && Convert.ToInt32(s.KeysAndQueries["id:Degree"]) == p).ToArray();
            if(sess.Count() == 0) continue;
            Array.Sort(sess, (a,b) => Convert.ToDouble(a.KeysAndQueries["id:Res"]).CompareTo(Convert.ToDouble(b.KeysAndQueries["id:Res"])));
            sess.ForEach(s => Console.WriteLine(s.Name));

            var Ref = new BoSSS.Application.LsTest.LevelSetUnitTests.LevelSetSwirlingFlowTest(2, p, false, 0.0); // to get the "Phi" Functions etc.
            var RefC = BoSSS.Application.LsTest.LevelSetUnitTests.LSTstObj2CtrlObj(Ref, int.MaxValue, LevelSetEvolution.None, BoSSS.Solution.XdgTimestepping.LevelSetHandling.None, hRes.Max(), 0);
            var grd = RefC.GridFunc(); // evaluate on the same grid

            Func<ITimestepInfo, double> errorFunctional = ts => {
                var PhiCG = ts.Fields.Single(f => f.Identification == "Phi");

                var PhiCGFine = new LevelSet(new Basis(grd, PhiCG.Basis.Degree), "PhiFine");
                PhiCGFine.ProjectFromForeignGrid(1.0, (SinglePhaseField)PhiCG); // transfer all to the fine grid

                LevelSetTracker LsTrk = new LevelSetTracker((GridData)PhiCGFine.GridDat, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new string[] { "A", "B" }, (ILevelSet)PhiCGFine);
                LsTrk.UpdateTracker(0.0);

                double d = 0.0;
                SpeciesId spcId = LsTrk.SpeciesIdS[0];
                var SchemeHelper = LsTrk.GetXDGSpaceMetrics(spcId, p).XQuadSchemeHelper;
                var vqs = SchemeHelper.GetLevelSetquadScheme(0, spcId, LsTrk.Regions.GetCutCellMask());
                CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
                    vqs.Compile(grd.iGridData, p),
                    delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
                        for(int j = 0; j < Length; j++ ){
                            var nodesG = MultidimensionalArray.Create(QR.NoOfNodes, grd.SpatialDimension);
                            grd.iGridData.TransformLocal2Global(QR.Nodes, nodesG, i0 + j);
                            for(int n = 0; n < QR.NoOfNodes; n++){
                                EvalResult[0, j, n] = (Math.Sqrt((nodesG[n, 0] - x0).Pow2() + (nodesG[n,1] - y0).Pow2()) - R).Pow2();
                            }
                        }
                    },
                    delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
                        for (int i = 0; i < Length; i++)
                            d += ResultsOfIntegration[i, 0];
                    }
                ).Execute();
                double err = Math.Sqrt(d);
                return err;
            };

            var errs = sess.Select(s => errorFunctional(s.Timesteps.Last())).ToArray();
            errs.ForEach(err => Console.WriteLine(err));

            var _plt = sess.ToGridConvergenceData(errorFunctional);
            _plt.dataGroups.Single().Name = " Degree: " + p;
            plt = plt.Merge(_plt);
        }        
        plts.Add(plt);
        plt.Title = "dt : " + dt + " Evo: " + Evo;
        plt.TitleFont = 30;
        plt.tmargin = 8;
        plt.ModFormat();
    }
}

In [ ]:
foreach(var plt in plts){
    display(plt.Title + " : ");
    plt.Regression().ForEach( g => display( g.Key + ", Slope : " + g.Value));
    display(plt.PlotNow());
    display(":::::::::::::::::::::::::::::::::::::::::");
}